In [23]:
import os 
os.chdir("/Users/mun-yeongmin/Downloads/") #working directory 설정

,useYm,prtAgNm,nm,intrvsslCo,intrvsslGrtg,fnshpCo,fnshpGrtg
0,202110,부산,일본,527,3130247,731,13328387.0
1,202110,부산,극동아시아 지역,479,5351746,1907,75368666.0
2,202110,부산,동남아시아 지역,29,330640,241,11067148.0
3,202110,부산,서남아시아 지역,0,0,3,89735.0
4,202110,부산,중동,0,0,35,2605509.0


In [33]:
import pandas as pd
cargo=pd.read_csv('경상남도 창원시_일반화물자동차운송사업자_20230620.csv', encoding="CP949")
cargo.head()

,업체명,면허종류,차량대수(직영),차량대수(위수탁),차량대수(확인불가),주소
0,개인사업자(업체명없음),일반화물,1.0,NaN,NaN,"경상남도 창원시 의창구 서곡길 133 (봉곡동,늘푸른마을 코오롱아파트)"
1,(주)해솔로지스,일반화물,NaN,2.0,NaN,"경상남도 창원시 의창구 사림로 47, 사림프라자 504호 (사림동)"
2,(주)창진로지스,일반화물,NaN,2.0,NaN,"경상남도 창원시 의창구 차상로 58, 한신빌딩 2층 202호 (팔용동)"
3,개인사업자(업체명없음),일반화물,2.0,NaN,NaN,"경상남도 창원시 의창구 서상로 27 (동정동,동정오성아파트)"
4,(주)우경특수,일반화물,NaN,2.0,NaN,"경상남도 창원시 의창구 차룡로48번길 44, 창원스마트업타워 S1807호 (팔용동)"


In [37]:
cargo.dtypes #NaN 0으로 대체하기 전 dtype 확인
#NaN 0값으로 변환
cargo['차량대수(직영)'] = pd.to_numeric(cargo['차량대수(직영)'], errors='coerce').fillna(0)
cargo['차량대수(위수탁)'] = pd.to_numeric(cargo['차량대수(위수탁)'], errors='coerce').fillna(0)
cargo['차량대수(확인불가)'] = pd.to_numeric(cargo['차량대수(확인불가)'], errors='coerce').fillna(0) 
#cargo.head()

,업체명,면허종류,차량대수(직영),차량대수(위수탁),차량대수(확인불가),주소
0,개인사업자(업체명없음),일반화물,1.0,0.0,0.0,"경상남도 창원시 의창구 서곡길 133 (봉곡동,늘푸른마을 코오롱아파트)"
1,(주)해솔로지스,일반화물,0.0,2.0,0.0,"경상남도 창원시 의창구 사림로 47, 사림프라자 504호 (사림동)"
2,(주)창진로지스,일반화물,0.0,2.0,0.0,"경상남도 창원시 의창구 차상로 58, 한신빌딩 2층 202호 (팔용동)"
3,개인사업자(업체명없음),일반화물,2.0,0.0,0.0,"경상남도 창원시 의창구 서상로 27 (동정동,동정오성아파트)"
4,(주)우경특수,일반화물,0.0,2.0,0.0,"경상남도 창원시 의창구 차룡로48번길 44, 창원스마트업타워 S1807호 (팔용동)"


In [38]:
# 1.구 단위 파싱 - 구 추출 함수
def extract_gu(address):
    gu_list = ['성산구', '의창구', '마산합포구', '마산회원구', '진해구']
    for gu in gu_list:
        if gu in str(address):
            return gu
    return '기타'

cargo['구'] = cargo['주소'].apply(extract_gu)
print(cargo['구'].value_counts())

구
진해구      135
의창구      134
마산회원구    121
성산구      103
마산합포구     73
Name: count, dtype: int64


In [39]:
#2.총 차량대수 파생변수 생성
cargo['총차량대수'] = cargo['차량대수(직영)'] + cargo['차량대수(위수탁)'] + cargo['차량대수(확인불가)']

# 구별 총 차량 집계
gu_summary = cargo.groupby('구')['총차량대수'].agg(['sum', 'count', 'mean']).reset_index()
gu_summary.columns = ['구', '총차량수', '업체수', '업체당평균차량']
print(gu_summary)
#업체 수는 제일 많은데 업체당 차량이 제일 적다 = 영세 소형 업체 집중

       구    총차량수  업체수   업체당평균차량
0  마산합포구   453.0   73  6.205479
1  마산회원구  1068.0  121  8.826446
2    성산구   506.0  103  4.912621
3    의창구   887.0  134  6.619403
4    진해구   580.0  135  4.296296


In [40]:
#3.진해구 vs 나머지 비교
cargo['진해여부'] = cargo['구'].apply(lambda x: '진해구' if x == '진해구' else '기타')

compare = cargo.groupby('진해여부').agg(
    업체수=('총차량대수', 'count'),
    총차량수=('총차량대수', 'sum'),
    평균차량수=('총차량대수', 'mean')
).reset_index()

print(compare)

  진해여부  업체수    총차량수     평균차량수
0   기타  431  2914.0  6.761021
1  진해구  135   580.0  4.296296


In [42]:
cargo['영세여부'] = cargo['총차량대수'].apply(
    lambda x: '영세(5대이하)' if x <= 5 else '중대형(6대이상)'
)

영세비율 = cargo.groupby(['구', '영세여부']).size().unstack(fill_value=0)
영세비율['영세비율(%)'] = (영세비율['영세(5대이하)'] / 
                          영세비율.sum(axis=1) * 100).round(1)
print(영세비율)

영세여부   영세(5대이하)  중대형(6대이상)  영세비율(%)
구                                  
마산합포구        69          4     94.5
마산회원구        96         25     79.3
성산구          86         17     83.5
의창구         102         32     76.1
진해구         115         20     85.2


In [ ]:
#진해구 전체라고 묶었을때 의미가 희석됨
#기존 항구-배후단지 거리: 5분 이내 혹은 10분 내
#기준: 차로 10분 거리 = 직선거리 약 5~6km